# ALL Grand Final Production Training (45/5)

이 노트북은 `ALL_LOSO.ipynb`의 함수/클래스를 재사용하여,
50명 전체 피험자를 `Train 45 / Val 5`로 분할해 최종 통합 모델을 학습하고 저장합니다.

- 샘플링레이트: `32Hz`
- 기본 정규화: `subject_zscore`
- 기본 손실함수: `focal`
- 저장 경로: `/home/binghin2/Myproject/Research/CATSA/Train/Individual_data/ALL/Save_model/ALL_Grand_Final_Production_Model.pt`

In [1]:
import argparse
import json
import random
from dataclasses import asdict
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import torch


def import_symbols_from_notebook(nb_path: Path, symbols: List[str]) -> Dict[str, object]:
    with nb_path.open("r", encoding="utf-8") as f:
        raw_nb = json.load(f)
    code_cells = [c for c in raw_nb.get("cells", []) if c.get("cell_type") == "code"]
    if len(code_cells) == 0:
        raise RuntimeError(f"No code cells found in {nb_path}")

    namespace: Dict[str, object] = {}
    # ALL_LOSO의 핵심 정의는 1번 코드 셀에 있으므로 그대로 실행해 재사용한다.
    first_source = code_cells[0].get("source", [])
    if isinstance(first_source, list):
        first_source = "".join(first_source)
    exec(first_source, namespace)

    missing = [name for name in symbols if name not in namespace]
    if missing:
        raise RuntimeError(f"Missing symbols from {nb_path.name}: {missing}")

    return {name: namespace[name] for name in symbols}


def build_arg_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description="Train final ALL production model with 45/5 subject split")
    parser.add_argument("--epochs", type=int, default=50)
    parser.add_argument("--batch_size", type=int, default=64)
    parser.add_argument("--lr", type=float, default=0.001)
    parser.add_argument("--norm_mode", type=str, default="subject_zscore", choices=["subject_zscore", "baseline_zscore", "none"])
    parser.add_argument("--loss_name", type=str, default="focal", choices=["focal", "weighted_bce"])
    parser.add_argument("--seed", type=int, default=42)
    return parser


def main(cli_args: Optional[List[str]] = None) -> Dict[str, object]:
    parser = build_arg_parser()
    if cli_args is None:
        args, _ = parser.parse_known_args()
    else:
        args = parser.parse_args(cli_args)

    loso_nb_path = Path("/home/binghin2/Myproject/Research/CATSA/Train/Individual_data/ALL/ALL_LOSO.ipynb")
    if not loso_nb_path.exists():
        raise FileNotFoundError(f"Cannot find source notebook: {loso_nb_path}")

    symbols = import_symbols_from_notebook(
        loso_nb_path,
        symbols=[
            "HyperParameters",
            "find_dataset_root",
            "discover_complete_subjects",
            "build_split_arrays",
            "make_loader",
            "MultiModal1DCNNTransformer",
            "make_criterion",
            "train_one_epoch",
            "evaluate",
            "FS_SLOW",
        ],
    )

    HyperParameters = symbols["HyperParameters"]
    find_dataset_root = symbols["find_dataset_root"]
    discover_complete_subjects = symbols["discover_complete_subjects"]
    build_split_arrays = symbols["build_split_arrays"]
    make_loader = symbols["make_loader"]
    MultiModal1DCNNTransformer = symbols["MultiModal1DCNNTransformer"]
    make_criterion = symbols["make_criterion"]
    train_one_epoch = symbols["train_one_epoch"]
    evaluate = symbols["evaluate"]
    FS_SLOW = int(symbols["FS_SLOW"])

    np.random.seed(args.seed)
    random.seed(args.seed)
    torch.manual_seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)

    dataset_root = find_dataset_root()
    all_subjects = discover_complete_subjects(dataset_root)
    if len(all_subjects) < 50:
        raise ValueError(f"Need 50 complete subjects, found {len(all_subjects)}")

    # 요구사항: 전체 50명 기준으로 45/5 split
    all_subjects = sorted(all_subjects, key=lambda x: int(x[3:]))[:50]
    val_subjects = sorted(random.sample(all_subjects, 5), key=lambda x: int(x[3:]))
    train_subjects = sorted([s for s in all_subjects if s not in val_subjects], key=lambda x: int(x[3:]))

    hp = HyperParameters(
        random_seed=args.seed,
        window_seconds=60,
        stride_seconds=10,
        stride_seconds_eval=60,
        batch_size=args.batch_size,
        epochs=args.epochs,
        learning_rate=args.lr,
        dropout_rate=0.3,
        weight_decay=1e-4,
        patience=8,
        transformer_heads=8,
        transformer_layers=2,
        transformer_ff_dim=256,
        normalization_mode=args.norm_mode,
        loss_name=args.loss_name,
        focal_gamma=2.0,
    )

    print("=== Grand Final ALL Training (45/5) ===")
    print(f"Sampling rate: {FS_SLOW}Hz")
    print(f"Window/Stride (seconds): {hp.window_seconds}/{hp.stride_seconds}")
    print(f"Normalization: {hp.normalization_mode}")
    print(f"Loss: {hp.loss_name}")
    print(f"Train subjects ({len(train_subjects)}): {train_subjects}")
    print(f"Val subjects ({len(val_subjects)}): {val_subjects}")

    x_train_acc, x_train_slow, y_train = build_split_arrays(
        dataset_root,
        train_subjects,
        hp,
        is_train=True,
    )
    x_val_acc, x_val_slow, y_val = build_split_arrays(
        dataset_root,
        val_subjects,
        hp,
        is_train=False,
    )

    train_loader = make_loader(x_train_acc, x_train_slow, y_train, hp.batch_size, shuffle=True)
    val_loader = make_loader(x_val_acc, x_val_slow, y_val, hp.batch_size, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = MultiModal1DCNNTransformer(hp=hp).to(device)
    criterion, loss_cfg = make_criterion(y_train, hp, device)
    optimizer = torch.optim.Adam(model.parameters(), lr=hp.learning_rate, weight_decay=hp.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=4)

    best_val_loss = float("inf")
    best_epoch = -1
    wait = 0
    best_final_state = None
    history: List[Dict[str, float]] = []

    for epoch in range(1, hp.epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_metrics = evaluate(model, val_loader, criterion, device)
        scheduler.step(val_metrics["loss"])

        history.append(
            {
                "epoch": float(epoch),
                "train_loss": float(train_loss),
                "val_loss": float(val_metrics["loss"]),
                "val_accuracy": float(val_metrics["accuracy"]),
                "val_f1": float(val_metrics["f1"]),
                "lr": float(optimizer.param_groups[0]["lr"]),
            }
        )

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_epoch = epoch
            wait = 0
            best_final_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= hp.patience:
                print(f"Early stopping at epoch {epoch} (best epoch: {best_epoch})")
                break

    if best_final_state is None:
        raise RuntimeError("best_final_state is None. Training did not produce a valid checkpoint.")

    save_dir = Path("/home/binghin2/Myproject/Research/CATSA/Train/Individual_data/ALL/Save_model")
    final_model_path = save_dir / "ALL_Grand_Final_Production_Model.pt"
    final_model_path.parent.mkdir(parents=True, exist_ok=True)

    payload = {
        "model_state_dict": best_final_state,
        "model_name": "ALL_Grand_Final_Production_Model",
        "created_from": "ALL_LOSO.ipynb",
        "sampling_rate_hz": FS_SLOW,
        "window_seconds": int(hp.window_seconds),
        "stride_seconds": int(hp.stride_seconds),
        "best_epoch": int(best_epoch),
        "best_val_loss": float(best_val_loss),
        "device_used_for_training": str(device),
        "hyperparameters": asdict(hp),
        "normalization_mode": hp.normalization_mode,
        "loss_config": loss_cfg,
        "subject_split": {"train": train_subjects, "val": val_subjects},
        "history": history,
    }

    torch.save(payload, final_model_path)

    print(f"Saved final production model to: {final_model_path}")
    return payload


# Notebook 실행 시 기본 인자 사용
final_result = main()
final_result

=== Grand Final ALL Training (45/5) ===
Sampling rate: 32Hz
Window/Stride (seconds): 60/10
Normalization: subject_zscore
Loss: focal
Train subjects (45): ['Sub1', 'Sub3', 'Sub4', 'Sub5', 'Sub6', 'Sub7', 'Sub9', 'Sub10', 'Sub11', 'Sub12', 'Sub13', 'Sub14', 'Sub15', 'Sub16', 'Sub17', 'Sub20', 'Sub21', 'Sub22', 'Sub23', 'Sub24', 'Sub26', 'Sub27', 'Sub28', 'Sub29', 'Sub30', 'Sub31', 'Sub32', 'Sub33', 'Sub34', 'Sub35', 'Sub36', 'Sub37', 'Sub38', 'Sub40', 'Sub41', 'Sub42', 'Sub43', 'Sub45', 'Sub46', 'Sub47', 'Sub48', 'Sub49', 'Sub50', 'Sub52', 'Sub53']
Val subjects (5): ['Sub2', 'Sub8', 'Sub18', 'Sub44', 'Sub51']
Early stopping at epoch 9 (best epoch: 1)
Saved final production model to: /home/binghin2/Myproject/Research/CATSA/Train/Individual_data/ALL/Save_model/ALL_Grand_Final_Production_Model.pt


{'model_state_dict': {'acc_branch.0.weight': tensor([[[ 0.1482,  0.1603, -0.0443,  0.1773, -0.0411,  0.0406, -0.0929,
             0.1151,  0.1724],
           [-0.1288,  0.1820,  0.0502,  0.1574,  0.0382,  0.1069, -0.0130,
             0.1637,  0.0430],
           [-0.0750,  0.0642, -0.0746, -0.0079, -0.0624,  0.1436, -0.1375,
            -0.0734, -0.0396]],
  
          [[-0.1073,  0.0261, -0.1815,  0.1821, -0.1547,  0.1568,  0.0402,
            -0.0543,  0.1271],
           [ 0.0428,  0.1680,  0.0339, -0.0470,  0.0642, -0.0404,  0.0928,
             0.1855,  0.1235],
           [-0.0722,  0.1224,  0.0470,  0.1097, -0.1057, -0.1786, -0.0624,
            -0.1352,  0.1701]],
  
          [[ 0.0610,  0.0857,  0.0654,  0.0026,  0.1569, -0.1305,  0.0182,
            -0.1257,  0.0653],
           [-0.0580,  0.0664, -0.0315,  0.1671, -0.1060, -0.1068, -0.1069,
             0.1806,  0.0722],
           [ 0.1845, -0.1578, -0.1889, -0.1497, -0.1287,  0.0784,  0.0698,
             0.1601, -0.09